In [2]:
import pandas as pd
import json

with open("/content/drive/MyDrive/data-sets/urls.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

df.to_csv("phishing_urls.csv", index=False)

print(df.head())
print(df.columns)

                                                text  label
0      http://webmail-brinkster.com/ex/?email=%20%0%      1
1                         billsportsmaps.com/?p=1206      0
2  www.sanelyurdu.com/language/homebank.tsbbank.c...      1
3                          ee-billing.limited323.com      1
4                   indiadaily.com/bolly_archive.htm      0
Index(['text', 'label'], dtype='object')


In [3]:
df=df.drop_duplicates()

In [4]:
!pip install tldextract

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 3.2 MB/s eta 0:00:00


In [5]:
import re
import math
import pandas as pd
from urllib.parse import urlparse
import tldextract
from collections import Counter

def has_ip_in_url(url):
    ip_pattern_v4 = r'\b(?:(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\b'
    ip_pattern_v6 = r'\b(?:[0-9a-fA-F]{1,4}:){7}[0-9a-fA-F]{1,4}\b'
    return bool(re.search(ip_pattern_v4, url) or re.search(ip_pattern_v6, url))

def calculate_entropy(s):
    if not s:
        return 0
    counts = Counter(s)
    probabilities = [c / len(s) for c in counts.values()]
    return -sum(p * math.log2(p) for p in probabilities)  # ✅ return inside function

def extract_domain_info(url):
    extracted = tldextract.extract(url)
    subdomain_count = len(extracted.subdomain.split('.')) if extracted.subdomain else 0
    domain_length = len(extracted.domain + '.' + extracted.suffix)
    tld = extracted.suffix
    return subdomain_count, domain_length, tld

def extract_url_parts_length(url):
    parsed = urlparse(url)
    return len(parsed.path), len(parsed.query)

def ratio_digits_to_letters(url):
    num_digits = sum(c.isdigit() for c in url)
    num_letters = sum(c.isalpha() for c in url)
    return num_digits / num_letters if num_letters > 0 else 0

# Basic counts
df['url_length']       = df['text'].apply(len)
df['num_dots']         = df['text'].apply(lambda x: x.count('.'))
df['num_hyphens']      = df['text'].apply(lambda x: x.count('-'))
df['num_underscores']  = df['text'].apply(lambda x: x.count('_'))
df['num_slashes']      = df['text'].apply(lambda x: x.count('/'))
df['num_digits']       = df['text'].apply(lambda x: sum(c.isdigit() for c in x))
df['num_special_chars']= df['text'].apply(lambda x: sum(not c.isalnum() and not c.isspace() for c in x))

# Boolean flags
df['has_ip_in_url']    = df['text'].apply(has_ip_in_url)
df['has_https']        = df['text'].apply(lambda x: x.startswith('https://'))
df['has_at_symbol']    = df['text'].apply(lambda x: '@' in x)
df['has_double_slash'] = df['text'].apply(lambda x: '//' in urlparse(x).path)  # ✅ fixed

# Domain info
df[['subdomain_count', 'domain_length', 'tld']] = df['text'].apply(
    lambda x: pd.Series(extract_domain_info(x))
)


# URL part lengths
df[['path_length', 'query_length']] = df['text'].apply(
    lambda x: pd.Series(extract_url_parts_length(x))
)

# Entropy & digit ratio
df['entropy']               = df['text'].apply(calculate_entropy)
df['ratio_digits_to_letters'] = df['text'].apply(ratio_digits_to_letters)

# Suspicious keywords
for keyword in ['login', 'secure', 'update', 'verify', 'account', 'bank']:
    df[f'has_keyword_{keyword}'] = df['text'].apply(lambda x, k=keyword: k in x.lower())

In [7]:

# import joblib
import joblib

# After train/test split:
tld_stats = df.groupby('tld')['label'].agg(['mean', 'count'])
tld_stats.columns = ['phishing_rate', 'count']

# Only keep TLDs with >= 30 samples
reliable_tlds = tld_stats[tld_stats['count'] >= 30]

# Create dictionary
tld_phishing_rate_dict = reliable_tlds['phishing_rate'].to_dict()
# Example: {'com': 0.45, 'xyz': 0.9965, 'gov': 0.0, 'cn': 0.9991, ...}

# Compute global mean
global_mean = df['label'].mean()
# Example: 0.47 (47% of training URLs are phishing)

# Save for later use
joblib.dump(tld_phishing_rate_dict, 'tld_phishing_rate.pkl')
joblib.dump(global_mean, 'global_mean.pkl')

['global_mean.pkl']

In [8]:
# Pattern 1 — Random gibberish subdomain (e.g. jfgybv.fbdffsa.cn)
def has_random_subdomain(url):
    extracted = tldextract.extract(url)
    subdomain = extracted.subdomain
    if not subdomain:
        return False
    # Check if subdomain parts look random (high consonant ratio, no vowels)
    parts = subdomain.split('.')
    for part in parts:
        vowels = sum(c in 'aeiou' for c in part.lower())
        if len(part) >= 5 and vowels / len(part) < 0.2:  # less than 20% vowels
            return True
    return False

# Pattern 2 — Legitimate domain embedded inside URL (e.g. co.jp.2012bao.cn)
def has_embedded_legitimate_domain(url):
    trusted = ['paypal', 'amazon', 'google', 'apple', 'microsoft',
               'facebook', 'netflix', 'battle.net', 'amazon.co.jp']
    path = url.lower()
    return any(brand in path for brand in trusted)

# Pattern 3 — Too many subdomains (e.g. www.saiconsacrd.co.jp.2012bao.cn)
def count_subdomains(url):
    extracted = tldextract.extract(url)
    return len(extracted.subdomain.split('.')) if extracted.subdomain else 0

# Add to dataframe
df['has_random_subdomain']        = df['text'].apply(has_random_subdomain)
df['has_embedded_legit_domain']   = df['text'].apply(has_embedded_legitimate_domain)
df['subdomain_count']             = df['text'].apply(count_subdomains)

In [9]:
# ❌ Problem
# 'https://www.google.com' → 'google' in url → True  (wrong!)

# ✅ Fix — only flag if the brand appears in subdomain/path but NOT as the actual domain
def has_embedded_legitimate_domain(url):
    trusted_brands = ['paypal', 'amazon', 'google', 'apple', 'microsoft',
                      'facebook', 'netflix', 'battlenet', 'ebay', 'bank']
    try:
        extracted = tldextract.extract(url)
        actual_domain = extracted.domain.lower()
        parsed = urlparse(url)

        # Check subdomain + path only, not the real domain itself
        subdomain = extracted.subdomain.lower()
        path = parsed.path.lower()

        for brand in trusted_brands:
            if brand in (subdomain + path) and brand != actual_domain:
                return True
    except:
        pass
    return False

In [10]:
# ✅ Fix — add typosquatting variants + check full URL string for lookalikes
def has_embedded_legitimate_domain(url):
    # Include common typosquatting variants
    trusted_brands = [
        'paypal', 'paypai', 'paypa1',
        'amazon', 'amazan', 'amazom', 'amazaon', 'arnazon',
        'google', 'gooogle', 'googie',
        'apple', 'appie', 'app1e',
        'microsoft', 'microsoft', 'mlcrosoft',
        'facebook', 'faceb00k', 'facebok',
        'netflix', 'netfl1x',
        'ebay', 'ebav',
        'bankofamerica', 'wellsfargo', 'chase',
    ]
    try:
        extracted = tldextract.extract(url)
        actual_domain = extracted.domain.lower()
        subdomain = extracted.subdomain.lower()
        parsed = urlparse(url)
        path = parsed.path.lower()
        check_string = subdomain + path  # don't include actual domain

        for brand in trusted_brands:
            if brand in check_string and brand != actual_domain:
                return True
    except:
        pass
    return False

In [11]:
# ✅ Better — check both subdomain AND domain for randomness
def has_random_subdomain(url):
    try:
        extracted = tldextract.extract(url)
        # Check subdomain parts + domain itself
        parts = extracted.subdomain.split('.') if extracted.subdomain else []
        parts.append(extracted.domain)  # also check domain name

        for part in parts:
            if len(part) >= 5:
                vowels = sum(c in 'aeiou' for c in part.lower())
                if vowels / len(part) < 0.2:  # less than 20% vowels = random
                    return True
    except:
        pass
    return False

## now all preprocessing is done train the model
* we use two models xgboost and lightgbm

In [12]:
X = df.drop(columns=['text', 'label','tld'])  # drop URL string + target
y = df['label']

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [14]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model2 = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42
)

In [15]:
model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

In [16]:
model2.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 308974, number of negative: 348147
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.208770 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1761
[LightGBM] [Info] Number of data points in the train set: 657121, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.470193 -> initscore=-0.119368
[LightGBM] [Info] Start training from score -0.119368


LGBMClassifier(learning_rate=0.05, n_estimators=300, random_state=42)

In [17]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,roc_auc_score

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(accuracy,precision,recall,f1)

0.9186698400910634 0.9407371617890669 0.8825616784303568 0.910721325715813


In [18]:
y_pred2 = model2.predict(X_test)

accuracy2 = accuracy_score(y_test, y_pred2)
precision2 = precision_score(y_test, y_pred2)
recall2 = recall_score(y_test, y_pred2)
f12 = f1_score(y_test, y_pred2)

print(accuracy2,precision2,recall2,f12)

0.922985616108984 0.9380538178660117 0.8952664637699929 0.9161608397169136


In [19]:
# ROC-AUC

y_pred_proba = model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_pred_proba)
y_pred_proba2 = model2.predict_proba(X_test)[:, 1]
roc_auc2 = roc_auc_score(y_test, y_pred_proba2)
print(roc_auc)

print(roc_auc2)


0.9737657016003474
0.9762658760105325


### Saving Trained Models with `joblib`

First, we'll save your trained `model` (XGBoost) and `model2` (LightGBM) to disk. This is crucial for deploying your models without retraining them every time.

In [20]:
import joblib

# Save the XGBoost model
joblib.dump(model, 'xgboost_phishing_model.joblib')
print("XGBoost model saved as 'xgboost_phishing_model.joblib'")

# Save the LightGBM model
joblib.dump(model2, 'lightgbm_phishing_model.joblib')
print("LightGBM model saved as 'lightgbm_phishing_model.joblib'")

XGBoost model saved as 'xgboost_phishing_model.joblib'
LightGBM model saved as 'lightgbm_phishing_model.joblib'


## wrapper function for deployment

In [ ]:
import re
import math
import tldextract
from urllib.parse import urlparse
from collections import Counter

def extract_url_features(url, tld_phishing_rate=None, global_mean=0.5):
    """
    Extract all URL features for phishing detection.

    Args:
        url (str): URL to extract features from
        tld_phishing_rate (dict): Mapping of TLD → phishing rate (computed from training)
        global_mean (float): Default phishing rate for unknown TLDs (0.0-1.0)

    Returns:
        dict: All numeric features (no strings), or None if error
    """

    if tld_phishing_rate is None:
        tld_phishing_rate = {}

    trusted_brands = [
        'paypal', 'paypai', 'paypa1',
        'amazon', 'amazan', 'amazom', 'amazaon', 'arnazon',
        'google', 'gooogle', 'googie',
        'apple', 'appie', 'app1e',
        'microsoft', 'mlcrosoft',
        'facebook', 'faceb00k', 'facebok',
        'netflix', 'netfl1x',
        'ebay', 'ebav',
        'bankofamerica', 'wellsfargo', 'chase'
    ]

    suspicious_keywords = [
        'login', 'secure', 'update', 'verify', 'account', 'bank'
    ]

    shorteners = [
        'bit.ly', 'tinyurl', 't.co', 'goo.gl', 'ow.ly',
        'short.link', 'buff.ly', 'adf.ly'
    ]

    features = {}

    try:
        # Handle URLs without scheme
        if not url.startswith(('http://', 'https://')):
            url = 'http://' + url

        parsed = urlparse(url)
        extracted = tldextract.extract(url)

        subdomain = extracted.subdomain.lower()
        domain = extracted.domain.lower()
        suffix = extracted.suffix.lower()
        url_lower = url.lower()

        # ─────────────────────────────────
        # Basic counts
        # ─────────────────────────────────

        features['url_length'] = len(url)
        features['num_dots'] = url.count('.')
        features['num_hyphens'] = url.count('-')
        features['num_underscores'] = url.count('_')
        features['num_slashes'] = url.count('/')
        features['num_digits'] = sum(c.isdigit() for c in url)
        features['num_special_chars'] = sum(
            not c.isalnum() and not c.isspace() for c in url
        )

        # ─────────────────────────────────
        # Boolean flags
        # ─────────────────────────────────

        # IP address detection
        ip_pattern_v4 = r'\b(?:(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\b'
        ip_pattern_v6 = r'\b(?:[0-9a-fA-F]{1,4}:){7}[0-9a-fA-F]{1,4}\b'

        features['has_ip_in_url'] = int(
            bool(re.search(ip_pattern_v4, url) or re.search(ip_pattern_v6, url))
        )

        features['has_https'] = int(url.startswith('https://'))
        features['has_at_symbol'] = int('@' in url)
        features['has_double_slash'] = int('//' in parsed.path)
        features['has_punycode'] = int('xn--' in url_lower)

        # ─────────────────────────────────
        # Domain info
        # ─────────────────────────────────

        features['subdomain_count'] = (
            len(subdomain.split('.')) if subdomain else 0
        )
        features['domain_length'] = len(extracted.domain + '.' + extracted.suffix)

        # ─────────────────────────────────
        # TLD Risk Features (NO RAW STRING)
        # ─────────────────────────────────

        tld_rate = tld_phishing_rate.get(suffix, global_mean)
        features['tld_phishing_rate'] = tld_rate

        # Binned risk tier (0-4)
        if tld_rate >= 0.99:
            features['tld_risk_tier'] = 4
        elif tld_rate >= 0.80:
            features['tld_risk_tier'] = 3
        elif tld_rate >= 0.40:
            features['tld_risk_tier'] = 2
        elif tld_rate >= 0.10:
            features['tld_risk_tier'] = 1
        else:
            features['tld_risk_tier'] = 0

        features['tld_is_trusted'] = int(tld_rate == 0.0)
        features['tld_is_high_risk'] = int(tld_rate >= 0.99)

        # ─────────────────────────────────
        # URL part lengths
        # ─────────────────────────────────

        features['path_length'] = len(parsed.path)
        features['query_length'] = len(parsed.query)

        # ─────────────────────────────────
        # Entropy (randomness score)
        # ─────────────────────────────────

        counts = Counter(url)
        probabilities = [count / len(url) for count in counts.values()]
        features['entropy'] = (
            -sum(p * math.log2(p) for p in probabilities) if url else 0
        )

        # ─────────────────────────────────
        # Digit-to-letter ratio
        # ─────────────────────────────────

        digits = sum(c.isdigit() for c in url)
        letters = sum(c.isalpha() for c in url)
        features['ratio_digits_to_letters'] = (
            digits / letters if letters > 0 else 0
        )

        # ─────────────────────────────────
        # Suspicious keywords
        # ─────────────────────────────────

        for keyword in suspicious_keywords:
            features[f'has_keyword_{keyword}'] = int(keyword in url_lower)

        # ─────────────────────────────────
        # Random domain/subdomain
        # ─────────────────────────────────

        features['has_random_subdomain'] = 0
        parts = subdomain.split('.') if subdomain else []
        parts.append(domain)

        for part in parts:
            if len(part) >= 5:
                vowels = sum(c in 'aeiou' for c in part)
                if vowels / len(part) < 0.2:  # < 20% vowels = random
                    features['has_random_subdomain'] = 1
                    break

        # ─────────────────────────────────
        # Embedded legitimate brand
        # ─────────────────────────────────

        features['has_embedded_legit_domain'] = 0
        check_string = subdomain + parsed.path.lower()

        for brand in trusted_brands:
            if brand in check_string and brand != domain:
                features['has_embedded_legit_domain'] = 1
                break

        # ─────────────────────────────────
        # URL shortener detection
        # ─────────────────────────────────

        features['is_shortened_url'] = int(
            any(shortener in url_lower for shortener in shorteners)
        )

    except Exception as e:
        # Return None on any error — caller must handle
        return None

    return features